# Explainable Pneumonia Detection — XAI Analysis

This notebook loads the trained CNN, evaluates its predictions on the test set, and uses **Grad-CAM** to inspect representative true-positive, true-negative, false-positive, and false-negative cases.

> **Interpretation note:** Grad-CAM shows regions associated with the model output; it does not establish clinical causality or diagnostic validity.

## 1. Configuration and imports

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from explainability.gradcam import make_gradcam_heatmap, overlay_gradcam

MODEL_PATH = Path('pneumonia_cnn_model.h5')
DATA_DIR = Path('chest_xray')
IMG_SIZE = 150
BATCH_SIZE = 32
CLASS_NAMES = ['NORMAL', 'PNEUMONIA']

assert MODEL_PATH.exists(), f'Model not found: {MODEL_PATH}'
assert (DATA_DIR / 'test').exists(), f'Test directory not found: {DATA_DIR / "test"}'

## 2. Load the trained model

In [ ]:
model = tf.keras.models.load_model(MODEL_PATH)
model.summary()
print('Input shape:', model.input_shape)
print('Output shape:', model.output_shape)

## 3. Load the test set deterministically

In [ ]:
def load_split(split_dir, max_per_class=None):
    images, labels, paths = [], [], []
    for label, class_name in enumerate(CLASS_NAMES):
        files = sorted(split_dir.joinpath(class_name).glob('*.jpeg'))
        files += sorted(split_dir.joinpath(class_name).glob('*.jpg'))
        if max_per_class is not None:
            files = files[:max_per_class]
        for path in files:
            img = load_img(path, target_size=(IMG_SIZE, IMG_SIZE), color_mode='rgb')
            images.append(img_to_array(img).astype('float32') / 255.0)
            labels.append(label)
            paths.append(path)
    return np.stack(images), np.asarray(labels), paths

X_test, y_test, test_paths = load_split(DATA_DIR / 'test')
print(f'Test images: {len(X_test):,}')
print('Class counts:', dict(zip(CLASS_NAMES, np.bincount(y_test))))

## 4. Classification performance

In [ ]:
scores = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1).reshape(-1)
y_pred = (scores >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))
print(f'ROC-AUC: {roc_auc_score(y_test, scores):.4f}')

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot()
plt.title('Test-set confusion matrix')
plt.show()

## 5. Select representative TP / TN / FP / FN cases

In [ ]:
case_indices = {
    'True Positive': np.where((y_test == 1) & (y_pred == 1))[0],
    'True Negative': np.where((y_test == 0) & (y_pred == 0))[0],
    'False Positive': np.where((y_test == 0) & (y_pred == 1))[0],
    'False Negative': np.where((y_test == 1) & (y_pred == 0))[0],
}

representatives = {}
for name, indices in case_indices.items():
    if len(indices):
        representatives[name] = int(indices[0])
        idx = int(indices[0])
        print(f'{name:15s} | true={CLASS_NAMES[y_test[idx]]:9s} | pred={CLASS_NAMES[y_pred[idx]]:9s} | score={scores[idx]:.4f} | {test_paths[idx].name}')
    else:
        print(f'{name:15s} | no example found')

## 6. Generate Grad-CAM explanations

In [ ]:
def show_gradcam(idx, title):
    image_rgb = np.uint8(np.clip(X_test[idx] * 255, 0, 255))
    heatmap, pneumonia_score, layer_name = make_gradcam_heatmap(
        X_test[idx:idx+1], model
    )
    overlay = overlay_gradcam(image_rgb, heatmap)
    prediction = CLASS_NAMES[int(pneumonia_score >= 0.5)]

    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    axes[0].imshow(image_rgb)
    axes[0].set_title(f'Original\nTrue: {CLASS_NAMES[y_test[idx]]}')
    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM')
    axes[2].imshow(overlay)
    axes[2].set_title(f'Prediction: {prediction}\nP(pneumonia)={pneumonia_score:.3f}')
    for ax in axes:
        ax.axis('off')
    fig.suptitle(f'{title} — explanation from {layer_name}')
    plt.tight_layout()
    plt.show()

for title, idx in representatives.items():
    show_gradcam(idx, title)

## 7. Explanation stability check

A useful XAI sanity check is to ask whether a small, controlled input change produces a wildly different explanation. The metric below compares Grad-CAM maps with cosine similarity after a +5% brightness perturbation. Higher similarity means the spatial explanation changed less under this perturbation. This is **not** a clinical explanation-quality metric.

In [ ]:
def cosine_similarity(a, b):
    a, b = a.reshape(-1), b.reshape(-1)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

for title, idx in representatives.items():
    original = X_test[idx:idx+1]
    perturbed = np.clip(original * 1.05, 0, 1)
    heat_a, _, _ = make_gradcam_heatmap(original, model)
    heat_b, _, _ = make_gradcam_heatmap(perturbed, model)
    similarity = cosine_similarity(heat_a, heat_b)
    print(f'{title:15s}: cosine similarity = {similarity:.4f}')

## 8. Interpretation checklist

When reviewing the heatmaps, ask:
- Does the model focus on plausible thoracic regions rather than borders, text, markers, or acquisition artifacts?
- Are false positives driven by visually similar patterns or by irrelevant image regions?
- Are false negatives associated with weak/subtle patterns?
- Are explanations reasonably stable under small perturbations?

These checks are useful for **model debugging and hypothesis generation**. They do not establish clinical safety or causal reasoning.